# Genetic Programming with Expression Trees in Julia

This notebook introduces **Genetic Programming (GP)** using the `ExprRules.jl` package.

Genetic Programming is an evolutionary method in which the candidate solutions are not fixed-length numerical vectors. Instead, individuals may be:

- mathematical expressions;
- programs;
- logical rules;
- decision structures;
- expression trees.

We will first learn how to define and generate valid expressions using a grammar. Then we will implement the main GP operators:

1. selection;
2. tree crossover;
3. tree mutation;
4. tree permutation;
5. evolutionary replacement.

Finally, we will use GP for two tasks:

- evolving an expression that approximates $\pi$;
- symbolic regression.

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain how a grammar defines the GP search space.
2. Generate and evaluate expression trees.
3. Interpret genotype and phenotype in GP.
4. Explain tree crossover and mutation.
5. Understand the role of tree depth and expression size.
6. Define a fitness function for symbolic expressions.
7. Apply GP to symbolic regression.
8. Investigate GP parameters using repeated stochastic runs.


## Packages

The following cells install and load the packages used in the original notebook.

If the packages are already installed in your Julia environment, the `Pkg.add` cells only need to be executed once.


In [1]:
import Pkg; Pkg.add("ExprRules")

   Resolving package versions...
    Updating `~/Library/CloudStorage/OneDrive-Personal/FSEGA/cursuri/2025-2026/erasmus ta/cod_ro_kl/Project.toml`
  [e7058ba0] + ExprRules v0.4.6
    Updating `~/Library/CloudStorage/OneDrive-Personal/FSEGA/cursuri/2025-2026/erasmus ta/cod_ro_kl/Manifest.toml`
  [1520ce14] + AbstractTrees v0.4.5
  [66dad0bd] - AliasTables v1.1.3
  [ec485272] + ArnoldiMethod v0.4.0
  [bbf7d656] + CommonSubexpressions v0.3.1
  [34da2185] + Compat v4.18.1
⌅ [864edb3b] ↓ DataStructures v0.19.6 ⇒ v0.18.22
  [e7058ba0] + ExprRules v0.4.6
⌃ [86223c79] + Graphs v1.13.1
  [d25df0c9] + Inflate v0.1.5
⌅ [2ab3a3ac] ↓ LogExpFunctions v1.0.1 ⇒ v0.3.29
⌅ [bac558e1] ↓ OrderedCollections v2.0.1 ⇒ v1.8.2
  [43287f4e] - PtrArrays v1.4.0
  [699a6c99] + SimpleTraits v0.9.6
  [90137ffa] + StaticArrays v1.9.20
  [1e83bf80] + StaticArraysCore v1.4.4
⌅ [2913bbd2] ↓ StatsBase v0.34.13 ⇒ v0.33.21
  [b4f28e30] + TikzGraphs v1.4.0
  [37f6aa50] + TikzPictures v3.5.1
  [39424ebd] + TreeView v0.5.1
⌃ 

In [2]:
import Pkg; Pkg.add("TreeView")


   Resolving package versions...
   Installed Graphs ─ v1.14.0
    Updating `~/Library/CloudStorage/OneDrive-Personal/FSEGA/cursuri/2025-2026/erasmus ta/cod_ro_kl/Project.toml`
  [39424ebd] + TreeView v0.5.1
    Updating `~/Library/CloudStorage/OneDrive-Personal/FSEGA/cursuri/2025-2026/erasmus ta/cod_ro_kl/Manifest.toml`
  [ec485272] + ArnoldiMethod v0.4.0
  [bbf7d656] + CommonSubexpressions v0.3.1
  [864edb3b] + DataStructures v0.19.6
  [86223c79] + Graphs v1.14.0
  [d25df0c9] + Inflate v0.1.5
  [692b3bcd] + JLLWrappers v1.8.0
  [b964fa9f] + LaTeXStrings v1.4.1
  [1914dd2f] + MacroTools v0.5.16
  [bac558e1] + OrderedCollections v2.0.1
⌅ [aea7be01] + PrecompileTools v1.2.1
  [21216c6a] + Preferences v1.5.2
  [699a6c99] + SimpleTraits v0.9.6
  [90137ffa] + StaticArrays v1.9.20
  [1e83bf80] + StaticArraysCore v1.4.4
  [10745b16] + Statistics v1.11.4
  [b4f28e30] + TikzGraphs v1.4.0
  [37f6aa50] + TikzPictures v3.5.1
  [39424ebd] + TreeView v0.5.1
  [6e34b625] + Bzip2_jll v1.0.9+0
⌃ [8342

In [3]:
import Pkg; Pkg.add("Distributions")


   Resolving package versions...
   Installed PDMats ─ v0.11.41
    Updating `~/Library/CloudStorage/OneDrive-Personal/FSEGA/cursuri/2025-2026/erasmus ta/cod_ro_kl/Project.toml`
  [31c24e10] + Distributions v0.25.131
    Updating `~/Library/CloudStorage/OneDrive-Personal/FSEGA/cursuri/2025-2026/erasmus ta/cod_ro_kl/Manifest.toml`
  [7d9f7c33] + Accessors v0.1.45
  [66dad0bd] + AliasTables v1.1.3
  [38540f10] + CommonSolve v0.2.14
  [a33af91c] + CompositionsBase v0.1.2
  [187b0558] + ConstructionBase v1.6.0
  [9a962f9c] + DataAPI v1.16.0
  [31c24e10] + Distributions v0.25.131
  [ffbed154] + DocStringExtensions v0.9.5
  [1a297f60] + FillArrays v1.17.0
  [a0844989] + Gamma v1.2.0
  [34004b35] + HypergeometricFunctions v0.3.30
  [3587e190] + InverseFunctions v0.1.17
  [92d709cd] + IrrationalConstants v0.2.6
  [2ab3a3ac] + LogExpFunctions v1.0.1
  [e1d29d7a] + Missings v1.2.0
  [90014a1f] + PDMats v0.11.41
  [43287f4e] + PtrArrays v1.4.0
  [1fd47b50] + QuadGK v2.11.3
  [189a3867] + Reexport

In [4]:
using ExprRules
using TreeView
using Distributions
using Random


┌ Warning: Failed to load PopplerBackend; falling back on DVIBackend
│   cause = Evaluation into the closed module `TikzPictures` breaks incremental compilation because the side effects will not be permanent. This is likely due to some other module mutating `TikzPictures` with `eval` during precompilation - don't do this.
└ @ TikzPictures ~/.julia/packages/TikzPictures/uIHWC/src/svg.jl:29


# Part I — Expression grammars and trees

## 1. Grammars

In Genetic Programming, a grammar specifies which expressions are valid.

A grammar consists of **production rules**. Each rule describes how a symbol can be expanded.

For example, if `R` represents a real-valued expression, rules may allow:

$$
R \rightarrow R \times A,
$$

$$
R \rightarrow f(R),
$$

or

$$
R \rightarrow c,
$$

where $c$ is a randomly generated constant.

The grammar therefore determines the **search space** of the evolutionary algorithm. If an operation is not included in the grammar, GP cannot discover an expression that uses it.


In [5]:
# Example 20.3. Example of defining a grammar using the ExprRules.jl package.
grammar = @grammar begin

    R = R * A # multiple children
    R = f(R) # call a function
    R =_(randn()) # random variable generated on node creation
    # R = 1 | 2 | 3 # equivalent to R = 1, R = 2, and R = 3
    # R = |(4:6) # equivalent to R = 4, R = 5, and R = 6
    A = 7 # rules for different return types
end;

f(x) = 2x

f (generic function with 1 method)

In [6]:
grammar

1: R = R * A
2: R = f(R)
3: R = _(randn())
4: A = 7


### Reading the grammar

The example above contains two return types, `R` and `A`.

The rules include:

- multiplication of an `R` expression by an `A` expression;
- application of the function `f`;
- a random Gaussian constant created when the node is generated;
- the constant value `7` for type `A`.

Notice that the grammar controls both the **syntactic structure** and the **types** of valid trees.


### Exercise 1 — Modify a grammar

Extend the grammar so that expressions of type `R` can also contain:

- addition;
- subtraction;
- the constant `1`.

Generate 10 random expressions using the modified grammar.

Inspect them and answer:

1. Which operators appear most often?
2. Are all generated expressions syntactically valid?
3. How does increasing the maximum depth affect expression complexity?


In [7]:
# Exercise 1

# Define an extended grammar here.
# Generate and display 10 random expressions.


## 2. Generating expression trees

`ExprRules.jl` represents expressions as trees.

An internal node usually represents an operator or function, while terminal nodes contain constants or variables.

For example,

$$
(x+2)\times x
$$

can be represented conceptually as

```text
        *
       / \
      +   x
     / \
    x   2
```

The command

```julia
rand(RuleNode, grammar, :R, depth)
```

creates a random tree compatible with the grammar.


In [8]:
rand(RuleNode, grammar, :R,6)

3,

In [9]:
[get_executable(rand(RuleNode, grammar, :R,6),  grammar)  for i in 1:10]

10-element Vector{Any}:
   :(f(f((f(-0.3923764098612923) * 7) * 7)))
   :(f(f((0.6788305284789906 * 7) * 7)) * 7)
  0.29075781404389534
   :(f(f(-1.8035175589262749)))
 -1.1031497173877711
   :(f(1.7095120920705607 * 7))
 -2.3980008390584646
   :(f(f(0.36753357110926355) * 7))
   :(f(1.3974786828351182))
   :(f(0.3085174405295682))

In [10]:
expr = rand(RuleNode, grammar, :R,40)

2{2{2{2{2{3}}}}}

In [11]:
get_executable(expr,  grammar)

:(f(f(f(f(f(-0.7714765591951664))))))

### Exercise 2 — Tree size and depth

Generate 100 random trees for each maximum depth

$$
d\in\{2,4,6,8\}.
$$

For each group compute:

- average tree size;
- minimum tree size;
- maximum tree size.

Create a plot of **average tree size versus maximum depth**.

**Question:** Why can unrestricted tree growth become a problem in Genetic Programming?


In [12]:
# Exercise 2

depths = [2, 4, 6, 8]

# Generate trees, measure their sizes, and summarize the results.


4-element Vector{Int64}:
 2
 4
 6
 8

## 3. From a tree to an executable expression

A GP individual has two useful representations:

- the **tree representation**, used by evolutionary operators;
- the **executable expression**, used to compute its output.

`get_executable` converts a `RuleNode` into a Julia expression.

To evaluate expressions that contain symbols or dynamically generated terms, the notebook uses a `SymbolTable`.


In [13]:

Symbols = SymbolTable(grammar)

Dict{Symbol, Any} with 2 entries:
  :f => f
  :* => *

In [14]:
Core.eval(Symbols, get_executable(expr,  grammar) )

-24.687249894245326

### Exercise 3 — Genotype and phenotype

Generate five random trees.

For each tree display:

1. the `RuleNode`;
2. its executable expression;
3. its evaluated numerical value.

In GP terminology, the tree is often considered the **genotype**, while its behavior or resulting expression represents the **phenotype**.

**Question:** Can two different trees produce the same numerical value?


In [16]:
# Exercise 3

# Generate five trees and display tree, executable expression, and value.


# Part II — Genetic Programming operators

A GP algorithm applies evolutionary operators directly to trees.

The overall cycle is similar to a Genetic Algorithm:

1. evaluate all trees;
2. select parents;
3. recombine subtrees;
4. mutate trees;
5. evaluate offspring;
6. retain improved individuals.

The important difference is that the solution representation has **variable structure and variable size**.


## 4. Selection

The notebook uses **truncation selection**.

If the population contains $m$ individuals and the parameter is `k`, only the best `k` individuals can become parents.

This creates strong selection pressure.

A small `k` strongly favors the best individuals, while a larger `k` maintains more diversity.


In [17]:
abstract type SelectionMethod end
struct TruncationSelection <: SelectionMethod
    k # top k to keep
end
function select(t::TruncationSelection, y)
    p = sortperm(y)
    return [p[rand(1:t.k, 2)] for i in y]

end

select (generic function with 1 method)

### Exercise 4 — Selection pressure

Suppose the population contains 20 individuals with fitness values

```julia
y = collect(1.0:20.0)
```

Compare:

```julia
TruncationSelection(2)
TruncationSelection(5)
TruncationSelection(10)
```

Perform 1000 parent selections for each case and count how often every individual is selected.

Plot the selection frequencies.

**Question:** How does `k` influence selection pressure and diversity?


In [18]:
# Exercise 4

y = collect(1.0:20.0)

# Compare different truncation-selection settings.


20-element Vector{Float64}:
  1.0
  2.0
  3.0
  4.0
  5.0
  6.0
  7.0
  8.0
  9.0
 10.0
 11.0
 12.0
 13.0
 14.0
 15.0
 16.0
 17.0
 18.0
 19.0
 20.0

## 5. Tree crossover

Tree crossover exchanges structural information between two parent expressions.

A subtree is selected from one parent and inserted into a compatible location in the other parent.

Because grammars may have multiple return types, crossover must preserve **type compatibility**.

The implementation also uses a maximum depth to prevent offspring from becoming arbitrarily large.


In [19]:
abstract type CrossoverMethod end
struct TreeCrossover <: CrossoverMethod
    grammar
    max_depth
end
function crossover(C::TreeCrossover, a, b)
    child = deepcopy(a)
    crosspoint = sample(b)
    typ = return_type(C.grammar, crosspoint.ind)
    d_subtree = depth(crosspoint)
    d_max = C.max_depth + 1 - d_subtree
    if d_max > 0 && contains_returntype(child, C.grammar, typ, d_max)
        loc = sample(NodeLoc, child, typ, C.grammar, d_max)
        insert!(child, loc, deepcopy(crosspoint))
    end
    child
end

crossover (generic function with 1 method)

In [20]:
expr=rand(RuleNode, grammar, :R,4)
cr= sample(expr)
expr,cr, cr.ind, get_executable(expr, grammar),get_executable(cr, grammar), return_type(grammar, cr.ind)

(3,, 3,, 3, 0.0643638842238073, 0.0643638842238073, :R)

### Exercise 5 — Visualize tree crossover

Generate two parent expressions with maximum depth 4.

Then:

1. display both executable expressions;
2. generate 20 children using `TreeCrossover`;
3. display the children;
4. compare their sizes with the parent sizes.

If `TreeView` functionality is available in your environment, visualize at least one parent and one child as trees.

**Question:** Does crossover always change the first parent?


In [21]:
# Exercise 5

# Generate two parents and several crossover children.
# Compare expression size and structure.


## 6. Tree mutation

Mutation introduces new genetic material.

The `TreeMutation` implementation:

1. copies the parent;
2. selects a random node;
3. determines the required return type at that position;
4. generates a new random subtree of that type;
5. inserts the new subtree.

The parameter $p$ controls the probability that mutation occurs.


In [22]:
abstract type MutationMethod end
struct TreeMutation <: MutationMethod
    grammar
    p
end
function mutate(M::TreeMutation, a)
    child = deepcopy(a)
    if rand() < M.p
        loc = sample(NodeLoc, child)
        typ = return_type(M.grammar, get(child, loc).ind)
        subtree = rand(RuleNode, M.grammar, typ)
        insert!(child, loc, subtree)
    end
    return child

end

mutate (generic function with 1 method)

### Exercise 6 — Mutation probability

Choose one parent tree and apply mutation 200 times for

$$
p\in\{0.05,0.25,0.5,1.0\}.
$$

For each value of $p$, estimate:

- the fraction of offspring that differ from the original;
- the mean offspring size;
- the maximum offspring size.

**Question:** Why might a very high mutation probability make evolutionary search unstable?


In [23]:
# Exercise 6

probabilities = [0.05, 0.25, 0.5, 1.0]

# Repeatedly mutate the same tree and collect statistics.


4-element Vector{Float64}:
 0.05
 0.25
 0.5
 1.0

## 7. Tree permutation

The second mutation operator in the notebook rearranges compatible child nodes.

This operation changes the structure of an expression without necessarily introducing completely new subtrees.

For commutative operations such as addition or multiplication, some permutations may produce an expression with identical behavior. For non-commutative operations such as subtraction or division, the effect can be much larger.


In [24]:
struct TreePermutation <: MutationMethod
    grammar
    p
end
function mutate(M::TreePermutation, a)
    child = deepcopy(a)
    if rand() < M.p
        node = sample(child)
        n = length(node.children)
        types = child_types(M.grammar, node)
        for i in 1:n-1
            c = 1
            for k in i+1:n
                if types[k] == types[i] &&
                   rand() < 1 / (c += 1)
                    node.children[i], node.children[k] =
                        node.children[k], node.children[i]

                end
            end
            
        end
    end
    return child
end

mutate (generic function with 2 methods)

### Exercise 7 — Mutation versus permutation

Generate one moderately complex tree.

Apply:

- `TreeMutation`;
- `TreePermutation`;

100 times each.

Compare:

1. how often the executable expression changes;
2. how much tree size changes;
3. how often the numerical result changes.

Explain the structural difference between the two operators.


In [25]:
# Exercise 7

# Compare TreeMutation and TreePermutation empirically.


# Part III — A complete GP algorithm

## 8. Evolutionary loop

The `genetic_algorithm` function evaluates all individuals and repeatedly:

1. selects parent pairs;
2. creates children with crossover;
3. mutates the children;
4. evaluates their fitness;
5. replaces a population member if the corresponding child is better.

The implementation minimizes the supplied fitness function.

The line

```julia
@show minimum(y)
```

prints the best fitness found in each generation and allows us to observe convergence.


In [26]:
function genetic_algorithm(f, population, max_iter, selection, crossover_, mutation)
    m = length(population)
    n = length(population[1])
    y = [f(population[i]) for i in 1:m]
    for i in 1:max_iter
        parents = select(selection, y)
        
        children = [crossover(crossover_, population[p[1]], population[p[2]]) for p in parents]
        children = [mutate(mutation, c) for c in children]
        children_y = [f(c) for c in children]
        for j in 1:m
            if children_y[j] < y[j]
                y[j] = children_y[j]
                population[j] = children[j]
            end
        end
        # population = children
        # y = children_y
        @show minimum(y)
    end
    return population[argmin(y)]
    
end

genetic_algorithm (generic function with 1 method)

### Exercise 8 — Store convergence history

Modify the GP algorithm so that it returns:

- the best tree;
- the best fitness value;
- a vector containing the best fitness after every generation.

Create a plot of best fitness versus generation.

Use a logarithmic vertical axis if it makes the convergence pattern easier to see.


In [27]:
# Exercise 8

function genetic_programming_with_history(f, population, max_iter,
                                          selection, crossover_, mutation)
    # Adapt the original algorithm.
end


genetic_programming_with_history (generic function with 1 method)

# Part IV — Evolving an approximation to $\pi$

The following example searches for an arithmetic expression whose numerical value is close to

$$
\pi.
$$

The grammar contains:

- integers from 1 to 9;
- addition;
- subtraction;
- multiplication;
- division.

The fitness contains two components:

$$
\text{fitness}
=
\log(|v-\pi|)
+
\frac{\text{tree size}}{1000}.
$$

The first term rewards numerical accuracy.

The second term penalizes large expressions. This is a simple form of **parsimony pressure**, used to discourage unnecessary growth.


In [28]:
grammar = @grammar begin
    R = |(1:9)
    R = R + R
    R = R - R
    R = R / R
    R = R * R
end

function f(node)
    value = Core.eval(node, grammar)
    if isinf(value) || isnan(value)
        return Inf
    end
    Δ = abs(value - π)
    return log(Δ) + length(node) / 1e3
end


population = [rand(RuleNode, grammar, :R) for i in 1:1000]
best_tree = genetic_algorithm(f, population, 30,
    TruncationSelection(50),
    TreeCrossover(grammar, 10),
    TreeMutation(grammar, 0.25))
    # TreePermutation(grammar, 0.25))
get_executable(best_tree, grammar)

minimum(y) = -1.9538009804964203
minimum(y) = -1.9538009804964203
minimum(y) = -1.9538009804964203
minimum(y) = -1.9538009804964203
minimum(y) = -1.9538009804964203
minimum(y) = -1.9538009804964203
minimum(y) = -1.9538009804964203
minimum(y) = -2.214859420961091
minimum(y) = -2.214859420961091
minimum(y) = -2.2168594209610912
minimum(y) = -2.2168594209610912
minimum(y) = -4.093795236407671
minimum(y) = -4.093795236407671
minimum(y) = -4.207813607761267
minimum(y) = -4.765649382705024
minimum(y) = -6.658086979553638
minimum(y) = -8.378625114241695
minimum(y) = -8.384625114241693
minimum(y) = -11.388713754938346
minimum(y) = -11.388713754938346
minimum(y) = -11.388713754938346
minimum(y) = -11.394713754938344
minimum(y) = -11.394713754938344
minimum(y) = -11.396713754938345
minimum(y) = -11.396713754938345
minimum(y) = -11.396713754938345
minimum(y) = -11.396713754938345
minimum(y) = -11.396713754938345
minimum(y) = -11.398713754938345
minimum(y) = -11.398713754938345


:(4 - (4 - ((5 - 8 / 7) / 4 + 3) / 7) / 4)

In [29]:
Core.eval(best_tree, grammar)

3.141581632653061

In [30]:
π

π = 3.1415926535897...

### Exercise 9 — Interpret the fitness function

Consider three hypothetical expressions with errors

$$
10^{-1},\quad 10^{-3},\quad 10^{-6}
$$

and tree sizes

$$
5,\quad 20,\quad 100.
$$

Compute the fitness contribution from:

1. the accuracy term;
2. the size penalty;
3. the complete fitness.

**Questions**

- Which term dominates in this implementation?
- What would happen if the size penalty were changed from `length(node)/1e3` to `length(node)/10`?


In [31]:
# Exercise 9

errors = [1e-1, 1e-3, 1e-6]
sizes = [5, 20, 100]

# Compute the two fitness components.


3-element Vector{Int64}:
   5
  20
 100

### Exercise 10 — Population size experiment

Run the $\pi$-approximation experiment using population sizes

$$
100,\;250,\;500,\;1000.
$$

Use several random seeds for each setting.

Record:

- best fitness;
- absolute error from $\pi$;
- tree size;
- runtime if desired.

**Question:** Does a larger population always produce a better expression for the same number of generations?


In [32]:
# Exercise 10

population_sizes = [100, 250, 500, 1000]

# Run repeated experiments and summarize the results.


4-element Vector{Int64}:
  100
  250
  500
 1000

### Exercise 11 — Control expression growth

Compare several maximum crossover depths, for example

$$
4,\;6,\;10,\;15.
$$

For each setting record:

- final best fitness;
- size of the best expression;
- maximum tree size observed.

Discuss the trade-off between **expressive power** and **bloat**.


In [33]:
# Exercise 11

depth_limits = [4, 6, 10, 15]

# Compare crossover depth limits.


4-element Vector{Int64}:
  4
  6
 10
 15

# Part V — Symbolic regression

Symbolic regression searches for both:

- the parameters of a model;
- the mathematical structure of the model itself.

Instead of assuming a fixed model such as

$$
y=\beta_0+\beta_1x+\beta_2x^2,
$$

GP evolves mathematical expressions directly.

The notebook defines a grammar containing:

- the variable $x$;
- multiplication;
- addition;
- subtraction;
- integer constants;
- $\sin(x)$.

A `SymbolTable` is used to assign numerical values to the variable $x$ during evaluation.


In [34]:
grammar = @grammar begin
    R = x
    R = R * R
    R = R + R
    R = R - R
    R = |(1:5)
    R = sin(x)
end

1: R = x
2: R = R * R
3: R = R + R
4: R = R - R
5: R = 1
6: R = 2
7: R = 3
8: R = 4
9: R = 5
10: R = sin(x)


In [35]:
const S = SymbolTable(grammar)
ground_truth(x) = x*x + 2x + 1

function loss(node)
    ex = get_executable(node, grammar)
    los = 0.0
    for x = -5.:1.:5.
        S[:x] = x
        los += abs2(Core.eval(S,ex) - ground_truth(x))
    end
    los + length(node)/1000
end

loss (generic function with 1 method)

In [36]:
population = [rand(RuleNode, grammar, :R) for i in 1:1000]
best_tree = genetic_algorithm(loss, population, 100,
    TruncationSelection(50),
    TreeCrossover(grammar, 10),
    TreeMutation(grammar, 0.25))
    # TreePermutation(grammar, 0.25))
get_executable(best_tree, grammar)

minimum(y) = 286.011
minimum(y) = 11.007
minimum(y) = 0.009
minimum(y) = 0.009
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y) = 0.007
minimum(y

:((x + 2) * x + 1)

### Understanding the loss

For every training value of $x$, the evolved expression is evaluated and compared with the target.

The original example uses

$$
\text{ground\_truth}(x)=x^2+2x+1.
$$

The error term is a sum of squared errors:

$$
\sum_i
\left(
\hat y_i-y_i
\right)^2.
$$

A small tree-size penalty is then added.

This means GP is simultaneously encouraged to find:

- accurate expressions;
- relatively compact expressions.


### Exercise 12 — Recover a known polynomial

Run the existing symbolic-regression example several times.

For each run:

1. display the best evolved expression;
2. report its training loss;
3. evaluate it on a finer grid of $x$-values;
4. compare its predictions with
   $$
   x^2+2x+1.
   $$

Plot the true function and evolved function on the same figure.

**Question:** Must GP discover exactly the expression `(x + 1)^2` to achieve zero prediction error?


In [37]:
# Exercise 12

# Run the symbolic-regression example and compare predictions graphically.


### Exercise 13 — Learn a more complex target

Modify the code to learn

$$
f(x)=x^3+\sin(x)-3x+7.
$$

You will need to check whether the grammar is expressive enough to represent the target.

In particular, consider whether the grammar can construct:

- $x^3$;
- $\sin(x)$;
- $-3x$;
- the constant $7$.

If necessary, modify the grammar.

Run the GP several times and report the best expression found.


In [38]:
# Exercise 13

ground_truth2(x) = x^3 + sin(x) - 3x + 7

# Modify the grammar if necessary and evolve an expression.


ground_truth2 (generic function with 1 method)

### Exercise 14 — Training and test points

The original loss evaluates expressions only at

$$
x=-5,-4,\ldots,5.
$$

Split the problem into:

- training points;
- test points not used during evolution.

For example, evolve on integer $x$-values and test on half-integer values.

Compute:

- training MSE;
- test MSE.

**Question:** Can an evolved expression fit the training points well but behave poorly between them?


In [39]:
# Exercise 14

# Define separate training and test grids.
# Evolve using only the training grid and evaluate on both.


### Exercise 15 — Grammar design experiment

Create three grammars:

1. a small grammar containing only arithmetic operators;
2. a medium grammar adding `sin`;
3. a richer grammar adding another nonlinear function such as `cos`.

Use the same symbolic-regression target and the same computational budget.

Compare:

- best training error;
- test error;
- expression size;
- variability across runs.

**Discussion:** Why can adding more operators make the problem both easier and harder?


In [40]:
# Exercise 15

# Define three grammars and compare them experimentally.


# Part VI — Experimental evaluation of GP

Like other evolutionary algorithms, GP is stochastic.

A single run is not enough to characterize performance.

For a meaningful experiment, use several random seeds and report both solution quality and structural complexity.


### Exercise 16 — Mini GP benchmarking study

Choose one symbolic-regression target.

Compare at least three GP configurations, such as:

- different mutation probabilities;
- different selection strengths;
- different crossover depth limits.

Use at least 20 runs per configuration.

Store the following information in a `DataFrame`:

- configuration;
- run;
- random seed;
- best training error;
- test error;
- expression size.

Then summarize the results using:

- mean;
- median;
- standard deviation;
- boxplots.

**Questions**

1. Which configuration gives the lowest error?
2. Which gives the smallest expressions?
3. Is the most accurate configuration also the most interpretable?


In [41]:
# Exercise 16

# using DataFrames

# Build a repeated-run experiment for several GP configurations.


# Summary

In this notebook you studied the main ideas of Genetic Programming.

## Expression representation

A grammar defines the legal structures that GP may evolve.

## Evolutionary operators

GP adapts standard evolutionary ideas to trees:

- selection chooses parents;
- crossover exchanges subtrees;
- mutation replaces subtrees;
- permutation rearranges compatible tree components.

## Fitness design

The fitness function determines what GP learns.

For symbolic expressions, it is often useful to combine:

$$
\text{prediction error}
+
\lambda \times \text{expression complexity}.
$$

This creates a trade-off between accuracy and interpretability.

## Symbolic regression

GP can evolve the mathematical form of a model directly rather than optimizing parameters of a pre-specified equation.

### Optional challenge

Create a noisy symbolic-regression dataset:

$$
y=x^2+2x+1+\varepsilon,
\qquad
\varepsilon\sim N(0,\sigma^2).
$$

Compare GP solutions for several noise levels.

Investigate whether stronger complexity penalties help prevent overly complicated expressions when the data are noisy.


In [42]:
# Optional challenge
